![lop](../../images/logo_diive1_128px.png)

<span style='font-size:40px; display:block;'>
<b>
    Seasonal-Trend Decomposition
</b>
</span>

---
**Notebook version**: `1` (Apr 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)  

</br>

# **Description**

Separate time series into:
- **Trend**: Long-term direction (ecosystem recovery, climate impacts)
- **Seasonal**: Recurring patterns (annual growing season, diurnal cycle)
- **Residual**: Noise and anomalies (measurement errors, anomalous events)

Supports multiple decomposition methods:
- **STL (default)**: Robust, handles gaps and non-stationary data
- **Classical**: Simple moving-average method
- **Harmonic**: Fourier-based frequency analysis

</br>

# **Imports**

In [ ]:
import importlib.metadata
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from diive.configs.exampledata import load_exampledata_parquet
from diive.pkgs.analyses.seasonaltrend import SeasonalTrendDecomposition
from diive.core.plotting.seasonaltrend import (
    plot_decomposition, plot_seasonal_strength_by_period
)

warnings.filterwarnings("ignore")
version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")

</br>

# **Load example data**

In [ ]:
# Load example eddy covariance data
df = load_exampledata_parquet()

# Use NEE (Net Ecosystem Exchange) - shows clear seasonal cycle
nee = df['NEE_CUT_REF_orig'].copy()

print(f"Data range: {nee.index[0]} to {nee.index[-1]}")
print(f"Series length: {len(nee)} records")
print(f"Valid data: {nee.notna().sum()} ({100*nee.notna().sum()/len(nee):.1f}%)")
print(f"\nData statistics:")
print(nee.describe())

</br>

# **Example 1: Quick Harmonic Decomposition**

Use harmonic (FFT-based) method for fast decomposition without missing value issues.

In [ ]:
# Use 2-year subset and interpolate for clean example
nee_subset = nee.loc['2015':'2016'].copy()
nee_subset = nee_subset.interpolate(method='linear').ffill().bfill()

print(f"Using subset: {len(nee_subset)} records")
print(f"Data range: {nee_subset.index[0]} to {nee_subset.index[-1]}")

# Create decomposition with harmonic method (fast)
decomp = SeasonalTrendDecomposition(
    nee_subset,
    method='harmonic',
    n_harmonics=10,
    verbose=True
)

print(f"\nDecomposition complete!")
print(f"Seasonality strength: {decomp.seasonality_strength:.4f}")

</br>

# **Access Components**

In [ ]:
# Access components (computed on first access, cached)
trend = decomp.trend
seasonal = decomp.seasonal
residual = decomp.residual

print(f"Trend shape: {trend.shape}")
print(f"Trend range: {trend.min():.3f} to {trend.max():.3f}")
print(f"\nSeasonal shape: {seasonal.shape}")
print(f"Seasonal std: {seasonal.std():.3f}")
print(f"\nResidual shape: {residual.shape}")
print(f"Residual std: {residual.std():.3f}")

</br>

# **Summary**

In [ ]:
print(decomp.summary())

</br>

# **Visualization: 4-Panel Decomposition**

In [ ]:
# Plot decomposition components
fig = plot_decomposition(
    decomp,
    figsize=(14, 10),
    show_residual_acf=False,
    title="NEE Seasonal-Trend Decomposition (Harmonic)"
)
plt.show()

</br>

# **Detrending and Deseasonalizing**

In [ ]:
# Remove trend (keep seasonal + residual)
detrended = decomp.detrend()
print(f"Detrended NEE std: {detrended.std():.3f}")
print("→ Seasonal cycle and noise without long-term drift")

# Remove seasonality (keep trend + residual)
deseasonalized = decomp.deseasonalize()
print(f"\nDeseasonalized NEE std: {deseasonalized.std():.3f}")
print("→ Long-term changes without seasonal pattern")

</br>

# **Multi-Scale Seasonality Analysis**

In [ ]:
# Test different seasonal periods
periods = [
    (7 * 48, '7 days'),
    (30 * 48, '~30 days'),
    (365 * 48, '~365 days')
]

print("Seasonality strength at different time scales:\n")
results = {}
for period, label in periods:
    try:
        decomp_test = SeasonalTrendDecomposition(
            nee_subset, method='harmonic', n_harmonics=5, verbose=False
        )
        strength = decomp_test.seasonality_strength
        results[label] = strength
        print(f"{label:15}: strength = {strength:.4f}")
    except Exception as e:
        print(f"{label:15}: Error - {str(e)[:40]}")

</br>

# **Quality-Weighted Decomposition**

In [ ]:
# Create quality flags (0-1, where 1 = best quality)
quality_flags = pd.Series(
    0.8 * np.ones(len(nee_subset)), index=nee_subset.index
)

# Lower quality in winter months
winter_months = [1, 2, 11, 12]
winter_mask = quality_flags.index.month.isin(winter_months)
quality_flags[winter_mask] = 0.5

print(f"Quality flag statistics:")
print(f"  Mean quality: {quality_flags.mean():.3f}")
print(f"  High quality (>0.7): {(quality_flags > 0.7).sum()} records")
print(f"  Low quality (<0.7): {(quality_flags <= 0.7).sum()} records")

# Decompose with quality weighting
print(f"\nDecomposing with quality-weighted fitting...")
decomp_quality = SeasonalTrendDecomposition(
    nee_subset,
    quality=quality_flags,
    method='harmonic',
    n_harmonics=10,
    quality_weighted=True,
    verbose=False
)
print(f"Seasonality strength: {decomp_quality.seasonality_strength:.4f}")

</br>

# **Anomaly Detection**

In [ ]:
# Find large residuals (potential anomalies)
residual_std = residual.std()
threshold = 2 * residual_std
anomalies = residual[residual.abs() > threshold]

print(f"Anomaly detection (>{threshold:.3f}):")
print(f"  Total anomalies: {len(anomalies)}")
print(f"  Percentage: {100*len(anomalies)/len(residual):.2f}%")

if len(anomalies) > 0:
    print(f"\n  Top 5 largest residuals:")
    top_anomalies = residual.abs().nlargest(5)
    for i, (idx, val) in enumerate(top_anomalies.items(), 1):
        sign = '+' if residual[idx] > 0 else '-'
        print(f"    {i}. {idx}: {sign}{abs(val):.3f}")

</br>

# **Reconstruction**

In [ ]:
# Reconstruct without residuals
reconstructed = decomp.reconstruct(keep_components=['trend', 'seasonal'])

print("Reconstruction comparison:")
print(f"\nOriginal series std: {nee_subset.std():.3f}")
print(f"Reconstructed (trend + seasonal) std: {reconstructed.std():.3f}")
print(f"Residual std: {residual.std():.3f}")
print(f"\nNoise removed: {(1 - reconstructed.std()/nee_subset.std())*100:.1f}%")

</br>

# **Use Cases**

## Trend Analysis (Ecosystem Recovery)
Extract trend to analyze long-term ecosystem changes.

## Anomaly Detection (Residual Analysis)
Examine residuals to identify unusual events (storms, equipment failures).

## Seasonal Pattern Understanding
Study seasonal components to understand recurring environmental patterns.

</br>

# **End of notebook**

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished {dt_string}")